# 08 — Spatial Subset & Download

Download only the pixels you need: define a bounding box over your study area,
stream just those pixels from a remote NISAR HDF5 file, and export as GeoTIFFs
that fit comfortably on a laptop.

This notebook demonstrates:
1. Flexible bbox input (tuple, GeoJSON, file)
2. Subsetting GCOV products with size estimation
3. Visualising the downloaded clips
4. Subsetting GUNW coherence (optional)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/08_subset_download.ipynb)

## 1. Install and Import

In [ ]:
# Pin fsspec/s3fs to avoid Colab conflicts, then force-reinstall only nice-sar
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0"
%pip install -q --force-reinstall --no-deps "git+https://github.com/bullocke/nice-sar.git@main"

## Colab Runtime Note

If you already imported `nice_sar` earlier in this Colab runtime, restart the
runtime after running the install cell, then run the notebook from the top.
Colab can keep the previously imported package in memory even after
`%pip install --upgrade`.

In [ ]:
%matplotlib inline

import logging
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from nice_sar.auth.earthdata import (
    get_granule_url,
    get_https_filesystem,
    get_s3_filesystem,
    login,
)
from nice_sar.io.bbox_parser import parse_bbox
from nice_sar.io.subset import subset_product
from nice_sar.io.geotiff import read_band
from nice_sar.preprocess.calibration import linear_to_db
from nice_sar.viz.display import percentile_stretch

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger(__name__)

## 2. Define Your Study Area

`subset_product()` accepts a bounding box in several formats:

| Format | Example |
|---|---|
| Tuple / list | `(-58.24, 4.40, -58.06, 4.57)` |
| CSV string | `"-58.24,4.40,-58.06,4.57"` |
| GeoJSON dict | `{"type": "Polygon", "coordinates": [...]}` |
| GeoJSON string | `'{"type": "Feature", ...}'` |
| File path | `"aoi.geojson"` or `"aoi.shp"` |

Below we use a GeoJSON polygon over **Guyana** tropical forest.

In [ ]:
# Guyana tropical forest AOI — GeoJSON polygon
guyana_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [-58.236822602728694, 4.568883563565226],
            [-58.236822602728694, 4.398774446109351],
            [-58.06241464374432, 4.398774446109351],
            [-58.06241464374432, 4.568883563565226],
            [-58.236822602728694, 4.568883563565226],
        ]
    ],
}

# parse_bbox extracts the bounding box from any supported input
bbox = parse_bbox(guyana_geojson)
logger.info("Parsed bbox: %s", bbox)

## 3. Authenticate with NASA Earthdata

`login()` checks environment variables, then `~/.netrc`, then prompts interactively.

In [ ]:
try:
    login()
except Exception as e:
    logger.error(
        "Authentication failed: %s\n"
        "Ensure credentials are configured via one of:\n"
        "  1. Environment variables: EARTHDATA_USERNAME / EARTHDATA_PASSWORD\n"
        "  2. ~/.netrc with: machine urs.earthdata.nasa.gov login <user> password <pass>\n"
        "  3. Interactive prompt (Colab / Jupyter only)",
        e,
    )
    raise

# Detect environment — S3 direct read on AWS us-west-2, HTTPS elsewhere
if os.environ.get("AWS_DEFAULT_REGION") == "us-west-2":
    fs = get_s3_filesystem()
    granule_access = "s3"
    logger.info("Using S3 direct access (us-west-2)")
else:
    fs = get_https_filesystem()
    granule_access = "https"
    logger.info("Using HTTPS streaming (outside us-west-2)")

## 4. Search for GCOV Granules

Find NISAR L2 GCOV products that intersect our Guyana study area.

In [ ]:
import earthaccess

results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=bbox,
    count=5,
)
logger.info("Found %d GCOV granules", len(results))

# Use the first granule
granule_url = get_granule_url(results[0], access=granule_access)
logger.info("Granule URL: %s", granule_url)

## 5. Subset & Download GCOV

`subset_product()` streams only the pixels within the bbox, estimates
the download size, and exports each polarization as a GeoTIFF.

Set `confirm=False` to skip the interactive prompt in notebooks.

In [ ]:
output_dir = Path("guyana_gcov_subset")

tif_paths = subset_product(
    source=granule_url,
    product="GCOV",
    bbox=guyana_geojson,
    frequency="A",
    polarizations=["HH", "HV"],
    output_dir=output_dir,
    filesystem=fs,
    confirm=False,  # non-interactive
)

for p in tif_paths:
    logger.info("Exported: %s (%.1f KB)", p.name, p.stat().st_size / 1024)

## 6. Visualise the Subset GeoTIFFs

Read the exported GeoTIFFs, convert GCOV linear power to dB, and display
with a percentile stretch.

In [ ]:
fig, axes = plt.subplots(1, len(tif_paths), figsize=(6 * len(tif_paths), 5))
if len(tif_paths) == 1:
    axes = [axes]

for ax, tif in zip(axes, tif_paths):
    data, meta = read_band(tif)
    db = linear_to_db(data)
    stretched = percentile_stretch(db, lower=2, upper=98)

    ax.imshow(stretched, cmap="gray")
    ax.set_title(tif.stem, fontsize=10)
    ax.axis("off")

plt.suptitle("GCOV Subset — Guyana", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Preprocess the Subset

Apply a Lee speckle filter and compute an HH/HV dB RGB composite on the
downloaded subset.

In [ ]:
from nice_sar.preprocess.filters import lee_filter
from nice_sar.viz.rgb import sar_false_color

# Read HH and HV subsets
hh_path = [p for p in tif_paths if "_HH_" in p.name][0]
hv_path = [p for p in tif_paths if "_HV_" in p.name][0]

hh_data, _ = read_band(hh_path)
hv_data, _ = read_band(hv_path)

# Apply Lee filter (5×5 window)
hh_filt = lee_filter(hh_data, win_size=5)
hv_filt = lee_filter(hv_data, win_size=5)

# Convert to dB for RGB
hh_db = linear_to_db(hh_filt)
hv_db = linear_to_db(hv_filt)

# HH/HV false-colour composite (R=HH, G=HV, B=HH/HV ratio)
rgb = sar_false_color(
    hh_db, hv_db,
    db_range=(-25, 0),
    method="ratio",
)

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(rgb)
ax.set_title("HH / HV False-Colour Composite — Guyana Subset", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Alternative Bbox Formats

Demonstrations of the other supported bbox input formats.

In [ ]:
# ── Tuple ─────────────────────────────────────────────────
bbox_tuple = (-58.24, 4.40, -58.06, 4.57)
logger.info("From tuple:   %s", parse_bbox(bbox_tuple))

# ── CSV string ────────────────────────────────────────────
bbox_csv = "-58.24,4.40,-58.06,4.57"
logger.info("From CSV:     %s", parse_bbox(bbox_csv))

# ── GeoJSON string ────────────────────────────────────────
import json

bbox_geojson_str = json.dumps(guyana_geojson)
logger.info("From GeoJSON: %s", parse_bbox(bbox_geojson_str))

# ── GeoJSON file ──────────────────────────────────────────
aoi_file = Path("guyana_aoi.geojson")
aoi_file.write_text(json.dumps({"type": "Feature", "geometry": guyana_geojson, "properties": {}}))
logger.info("From file:    %s", parse_bbox(str(aoi_file)))

## 9. (Optional) Subset GUNW Coherence

The same `subset_product()` function works for GUNW products.
Specify `layers=["coherenceMagnitude"]` to download just the coherence band.

In [ ]:
# Uncomment and run if GUNW data is available for this area

# gunw_results = earthaccess.search_data(
#     short_name="NISAR_L2_GUNW_BETA_V1",
#     bounding_box=bbox,
#     count=2,
# )
# if gunw_results:
#     gunw_url = get_granule_url(gunw_results[0], access=granule_access)
#     coh_paths = subset_product(
#         source=gunw_url,
#         product="GUNW",
#         bbox=guyana_geojson,
#         frequency="A",
#         polarizations=["HH"],
#         layers=["coherenceMagnitude"],
#         output_dir=Path("guyana_gunw_subset"),
#         filesystem=fs,
#         confirm=False,
#     )
#     if coh_paths:
#         coh, _ = read_band(coh_paths[0])
#         fig, ax = plt.subplots(figsize=(7, 6))
#         im = ax.imshow(coh, cmap="RdYlGn", vmin=0, vmax=1)
#         plt.colorbar(im, ax=ax, label="Coherence")
#         ax.set_title("GUNW Coherence — Guyana Subset")
#         ax.axis("off")
#         plt.show()

## 10. CLI Usage

The same workflow is available from the command line:

```bash
# GCOV — download HH and HV for a bounding box
nice-sar subset \
    --bbox "-58.24,4.40,-58.06,4.57" \
    --product GCOV \
    --polarization HH --polarization HV \
    --start 2025-01-01 --end 2025-06-01 \
    --max-granules 3 \
    -o ./guyana_gcov/

# GCOV — from a GeoJSON file
nice-sar subset \
    --bbox-file aoi.geojson \
    --product GCOV \
    -o ./my_subset/

# GUNW coherence
nice-sar subset \
    --bbox "-58.24,4.40,-58.06,4.57" \
    --product GUNW \
    --layer coherenceMagnitude \
    -o ./guyana_gunw/
```

## Cleanup

Remove downloaded subsets and temporary files.

In [ ]:
# Uncomment to remove outputs
# import shutil
# shutil.rmtree("guyana_gcov_subset", ignore_errors=True)
# shutil.rmtree("guyana_gunw_subset", ignore_errors=True)
# Path("guyana_aoi.geojson").unlink(missing_ok=True)